<a href="https://colab.research.google.com/github/19mddill/Machine_Learning_Notebooks/blob/main/ensemble_learning_and_random_forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier,VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

In [5]:
X,y = make_moons(n_samples=500,noise=0.3,random_state=42)
X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42)

In [6]:
voting_clf = VotingClassifier(
    estimators=[
        ('lr',LogisticRegression()),
        ('rf',RandomForestClassifier()),
        ('svc',SVC())
    ]
)

In [7]:
voting_clf.fit(X_train,y_train)

VotingClassifier(estimators=[('lr', LogisticRegression()),
                             ('rf', RandomForestClassifier()), ('svc', SVC())])

In [8]:
for name,clf in voting_clf.named_estimators_.items():
    print(name,"=",clf.score(X_test,y_test))

lr = 0.864
rf = 0.904
svc = 0.896


In [9]:
voting_clf.predict(X_test[:1])

array([1])

In [10]:
[clf.predict(X_test[:1]) for clf in voting_clf.estimators_]

[array([1]), array([1]), array([0])]

In [11]:
voting_clf.score(X_test,y_test)

0.912

In [12]:
voting_clf.voting = 'soft'
voting_clf.named_estimators["svc"].probability = True
voting_clf.fit(X_train,y_train)


VotingClassifier(estimators=[('lr', LogisticRegression()),
                             ('rf', RandomForestClassifier()),
                             ('svc', SVC(probability=True))],
                 voting='soft')

In [13]:
for name,clf in voting_clf.named_estimators_.items():
    print(name,"=",clf.score(X_test,y_test))

lr = 0.864
rf = 0.896
svc = 0.896


In [14]:
print("\n=== Probabilities for X_test[0] ===")
print(f"True label: {y_test[0]}")
for name, clf in zip(voting_clf.named_estimators_, voting_clf.estimators_):
    proba = clf.predict_proba(X_test[:1])[0]
    print(f"{name}: class0={proba[0]:.3f}, class1={proba[1]:.3f}")


=== Probabilities for X_test[0] ===
True label: 1
lr: class0=0.499, class1=0.501
rf: class0=0.370, class1=0.630
svc: class0=0.575, class1=0.425


In [15]:
X_test[:1]

array([[0.50169252, 0.21717211]])

In [16]:
[clf.predict_proba(X_test[:1])[0]
                       for clf in voting_clf.estimators_]

[array([0.49900001, 0.50099999]),
 array([0.37, 0.63]),
 array([0.5751062, 0.4248938])]

In [17]:
import numpy as np
np.array([clf.predict_proba(X_test[:1])[0]
                       for clf in voting_clf.estimators_])

array([[0.49900001, 0.50099999],
       [0.37      , 0.63      ],
       [0.5751062 , 0.4248938 ]])

In [18]:

print("\n=== Averaged Probabilities (soft voting logic) ===")
all_probas = np.array([clf.predict_proba(X_test[:1])[0]
                       for clf in voting_clf.estimators_])
avg_proba = all_probas.mean(axis=0)
print(f"class0 avg={avg_proba[0]:.3f}, class1 avg={avg_proba[1]:.3f}")
print(f"Soft vote picks: class {np.argmax(avg_proba)}")


=== Averaged Probabilities (soft voting logic) ===
class0 avg=0.481, class1 avg=0.519
Soft vote picks: class 1


In [19]:
avg_proba

array([0.48136874, 0.51863126])

In [20]:

print(f"\nEnsemble prediction: {voting_clf.predict(X_test[:1])[0]}")
print(f"Ensemble score: {voting_clf.score(X_test, y_test):.4f}")


Ensemble prediction: 1
Ensemble score: 0.9120


# Bagging

In [21]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

In [22]:
X,y = make_moons(n_samples=500,noise=0.3,random_state=42)
X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42)

In [23]:
X_train.size

750

In [24]:
bag_clf = BaggingClassifier(
    DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=100,
    bootstrap=True,
    n_jobs=-1,
    oob_score=True
)

In [25]:
bag_clf.fit(X_train,y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), max_samples=100,
                  n_estimators=500, n_jobs=-1, oob_score=True)

In [26]:
bag_clf.score(X_test,y_test)

0.92

In [27]:
bag_clf.oob_score_

0.9226666666666666

In [28]:
from sklearn.metrics import accuracy_score
y_pred = bag_clf.predict(X_test)
accuracy_score(y_test,y_pred)

0.92

In [29]:
bag_clf.oob_decision_function_[:3]

array([[0.32731959, 0.67268041],
       [0.43010753, 0.56989247],
       [0.99748111, 0.00251889]])

# Random Forest

In [30]:
from sklearn.ensemble import RandomForestClassifier

In [31]:
rnd_clf = RandomForestClassifier(n_estimators=500,max_leaf_nodes=16,n_jobs=-1,random_state=42)

In [32]:
rnd_clf.fit(X_train,y_train)

RandomForestClassifier(max_leaf_nodes=16, n_estimators=500, n_jobs=-1,
                       random_state=42)

In [33]:
y_pred_rf = rnd_clf.predict(X_test)

In [34]:
bag_clf = BaggingClassifier(
    DecisionTreeClassifier(max_features="sqrt",max_leaf_nodes=16),
    n_estimators=500,
    n_jobs = -1,
    random_state=42
)